# AI文章検出ツール ― Binoculars vs 統計的手法 比較ノートブック

このノートブックでは、学生のレポートなどのテキストがAIで生成されたものかを **段落ごと** に判定します。

| 手法 | 概要 |
|---|---|
| **統計的手法** | 文長均一性・語彙多様性・接続詞頻度・句読点パターンの4指標 |
| **Binoculars** | 2つのLLM（Observer / Performer）のパープレキシティ比率 |

**実行環境**: Google Colab（T4 GPU 推奨、CPUでもフォールバック動作）

**使い方**: 全セルを順番に実行すると、最後にWeb UIが起動します。

## 1. セットアップ

In [ ]:
!pip install -q transformers torch accelerate bitsandbytes sentencepiece protobuf gradio

import warnings
warnings.filterwarnings("ignore")
print("セットアップ完了")

## 2. テキスト入力・段落分割

In [ ]:
import re

# ── ここにテキストを貼り付け ──
sample_text = """
人工知能の発展は、現代社会に大きな影響を与えている。特に、自然言語処理の分野では、大規模言語モデルの登場により、テキスト生成の品質が飛躍的に向上した。これにより、教育現場においても、AIが生成した文章と人間が書いた文章を区別することが重要な課題となっている。

一方で、従来の文章作成においては、個人の経験や感情が反映されることが多く、文体にも独自の特徴が現れる。例えば、句読点の打ち方や接続詞の使い方には、書き手の癖が出やすい。こうした特徴は、AIが生成する文章には見られにくい傾向がある。

昨日友達とカフェに行って、めっちゃ美味しいケーキ食べた。チョコレートのやつが最高だった！また行きたいな〜。

さらに、AI生成テキストの特徴として、文の長さが均一であること、語彙の選択が平均的であること、そして論理展開が整然としていることが挙げられる。したがって、これらの統計的特徴を分析することで、AI生成テキストを検出できる可能性がある。加えて、近年の研究では、言語モデル自体を用いた検出手法も提案されている。

つまり、教育現場では複数の検出手法を組み合わせることが望ましい。具体的には、統計的分析とAIモデルベースの分析を併用することで、より信頼性の高い判定が可能になると考えられる。
"""

# ── 段落分割 ──
MIN_PARAGRAPH_LENGTH = 50

def split_paragraphs(text):
    normalized = text.replace("\r\n", "\n").strip()
    if not normalized:
        return []
    chunks = re.split(r"\n\s*\n+", normalized)
    if len(chunks) == 1:
        chunks = normalized.split("\n")
    result = []
    for chunk in chunks:
        t = chunk.strip()
        if not t:
            continue
        result.append({"index": len(result), "text": t, "skipped": len(t) < MIN_PARAGRAPH_LENGTH})
    return result

paragraphs = split_paragraphs(sample_text)
for p in paragraphs:
    status = "SKIP" if p["skipped"] else "OK"
    print(f'[段落{p["index"]+1}] {status} ({len(p["text"])}文字): {p["text"][:40]}...')
print(f"\n合計: {len(paragraphs)}段落 (分析対象: {sum(1 for p in paragraphs if not p['skipped'])})")

## 3. 統計的手法（Webアプリ移植版）

| 指標 | AI的なテキストの特徴 |
|---|---|
| 文長均一性 | 文の長さのばらつきが小さい |
| 語彙多様性 (TTR) | TTRが中程度（0.4〜0.6付近） |
| 接続詞頻度 | 接続詞を多用する |
| 句読点規則性 | 句読点の間隔が等間隔に近い |

In [ ]:
import numpy as np

CONJUNCTIONS = [
    "また", "さらに", "一方で", "しかし", "そのため",
    "したがって", "つまり", "すなわち", "加えて", "それに加えて",
    "具体的には", "例えば", "このように", "以上のことから", "結果として",
    "特に", "なお", "ただし", "もっとも", "それゆえ",
]

def split_sentences(text):
    return [s.strip() for s in re.split(r"[。！？\n]+", text) if s.strip()]

def tokenize(text):
    tokens = re.findall(r"[a-zA-Z0-9]+", text)
    japanese = re.sub(r"[a-zA-Z0-9\s\u0000-\u007F]", "", text)
    for i in range(len(japanese) - 1):
        tokens.append(japanese[i:i+2])
    return tokens

def analyze_sentence_length_uniformity(sentences):
    if len(sentences) < 2:
        return 0.5
    lengths = [len(s) for s in sentences]
    mean = np.mean(lengths)
    if mean == 0:
        return 0.5
    cv = np.std(lengths) / mean
    return float(max(0, min(1, 1 - cv / 0.8)))

def analyze_vocabulary_diversity(text):
    tokens = tokenize(text)
    if len(tokens) < 10:
        return 0.5
    ttr = len(set(tokens)) / len(tokens)
    distance = abs(ttr - 0.5)
    return float(max(0, min(1, 1 - distance / 0.35)))

def analyze_conjunction_frequency(text, sentences):
    if not sentences:
        return 0.5
    count = sum(len(re.findall(c, text)) for c in CONJUNCTIONS)
    rate = count / len(sentences)
    return float(max(0, min(1, rate / 0.5)))

def analyze_punctuation_regularity(text):
    positions = [i for i, ch in enumerate(text) if ch in "、，"]
    if len(positions) < 3:
        return 0.5
    gaps = np.diff(positions)
    mean = np.mean(gaps)
    if mean == 0:
        return 0.5
    cv = np.std(gaps) / mean
    return float(max(0, min(1, 1 - cv / 0.8)))

def analyze_statistically(text):
    sentences = split_sentences(text)
    slu = analyze_sentence_length_uniformity(sentences)
    vd = analyze_vocabulary_diversity(text)
    cf = analyze_conjunction_frequency(text, sentences)
    pr = analyze_punctuation_regularity(text)
    overall = (slu + vd + cf + pr) / 4
    return {"slu": slu, "vd": vd, "cf": cf, "pr": pr, "overall": overall}

# テスト
for p in paragraphs:
    if p["skipped"]:
        continue
    r = analyze_statistically(p["text"])
    print(f'段落{p["index"]+1}: 統計スコア={r["overall"]:.3f}')
    print(f'  文長均一性={r["slu"]:.3f}, 語彙多様性={r["vd"]:.3f}, 接続詞={r["cf"]:.3f}, 句読点={r["pr"]:.3f}')
    print()

## 4. Binoculars（LLMベース検出）

**原理**: Observer（汎用LLM）とPerformer（instruction-tuned LLM）のクロスエントロピー比率で判定

In [ ]:
import torch

def get_device():
    if torch.cuda.is_available():
        name = torch.cuda.get_device_name(0)
        props = torch.cuda.get_device_properties(0)
        vram = getattr(props, "total_memory", getattr(props, "total_mem", 0)) / 1e9
        print(f"GPU検出: {name} (VRAM: {vram:.1f} GB)")
        return "cuda", vram
    print("GPU未検出 → CPUモード（軽量モデルを使用）")
    return "cpu", 0

DEVICE, VRAM = get_device()
USE_LARGE = DEVICE == "cuda" and VRAM > 10

if USE_LARGE:
    # Mistral-7B は transformers と互換性が高く安定動作する
    OBSERVER_NAME = "mistralai/Mistral-7B-v0.1"
    PERFORMER_NAME = "mistralai/Mistral-7B-Instruct-v0.1"
    print(f"大規模モデル: {OBSERVER_NAME} / {PERFORMER_NAME} (4bit量子化)")
else:
    OBSERVER_NAME = "openai-community/gpt2"
    PERFORMER_NAME = "openai-community/gpt2-medium"
    print(f"軽量モデル: {OBSERVER_NAME} / {PERFORMER_NAME}")

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

def load_model(name):
    print(f"ロード中: {name} ...", end=" ", flush=True)
    kwargs = {"device_map": "auto"} if DEVICE == "cuda" else {}
    if USE_LARGE:
        bnb = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_quant_type="nf4",
        )
        model = AutoModelForCausalLM.from_pretrained(
            name, quantization_config=bnb, trust_remote_code=True, **kwargs
        )
    else:
        model = AutoModelForCausalLM.from_pretrained(name, **kwargs)
        if DEVICE == "cuda":
            model = model.half()
    model.eval()
    tok = AutoTokenizer.from_pretrained(name, trust_remote_code=True)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    print("完了")
    return model, tok

observer_model, observer_tok = load_model(OBSERVER_NAME)
performer_model, performer_tok = load_model(PERFORMER_NAME)

In [ ]:
def compute_ce(model, tokenizer, text, max_length=512):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=max_length)
    ids = inputs["input_ids"].to(model.device)
    if ids.shape[1] < 2:
        return float("nan")
    with torch.no_grad():
        out = model(ids, labels=ids)
    return out.loss.item()

def binoculars_score(text):
    obs = compute_ce(observer_model, observer_tok, text)
    perf = compute_ce(performer_model, performer_tok, text)
    if np.isnan(obs) or np.isnan(perf) or perf == 0:
        return 0.5
    return obs / perf

def normalize_bino(raw):
    # raw ≈ 1.0 → AI, raw > 1.3 → 人間
    LOW, HIGH = 0.9, 1.4
    prob = 1.0 - (raw - LOW) / (HIGH - LOW)
    return float(max(0.0, min(1.0, prob)))

print("Binocularsスコア計算中...\n")
for p in paragraphs:
    if p["skipped"]:
        continue
    raw = binoculars_score(p["text"])
    norm = normalize_bino(raw)
    print(f'段落{p["index"]+1}: 生スコア={raw:.4f}, AI確率={norm:.3f}')

## 5. Web UI（Gradio）

下のセルを実行すると、テキストを入力して結果を表示できるWebインターフェースが起動します。

`share=True` により公開URLが生成され、他の人にも共有できます。

In [ ]:
import gradio as gr

def analyze_text(text):
    """テキストを分析してHTML結果を返す"""
    if not text or not text.strip():
        return "テキストを入力してください。"

    paras = split_paragraphs(text)
    if not paras:
        return "段落が見つかりませんでした。"

    to_analyze = [p for p in paras if not p["skipped"]]
    if not to_analyze:
        return "分析可能な段落がありません（50文字以上の段落を含めてください）。"

    # 各段落を分析
    results = []
    for p in paras:
        if p["skipped"]:
            results.append({
                "index": p["index"],
                "text": p["text"],
                "stat": None,
                "bino": None,
                "combined": None,
                "label": "スキップ",
                "skipped": True,
            })
            continue

        stat = analyze_statistically(p["text"])
        braw = binoculars_score(p["text"])
        bprob = normalize_bino(braw)
        comb = bprob * 0.7 + stat["overall"] * 0.3

        if comb >= 0.75:
            label = "AI"
        elif comb >= 0.40:
            label = "混在"
        else:
            label = "人間"

        results.append({
            "index": p["index"],
            "text": p["text"],
            "stat": stat["overall"],
            "bino": bprob,
            "combined": comb,
            "label": label,
            "skipped": False,
        })

    # HTML出力を生成
    def fmt(v):
        return "-" if v is None else f"{v:.0%}"

    def color(v):
        if v is None:
            return "#f3f4f6", "#9ca3af"
        if v >= 0.75:
            return "#fecaca", "#ef4444"
        if v >= 0.40:
            return "#fef08a", "#eab308"
        return "#bbf7d0", "#22c55e"

    def label_text(v):
        if v is None:
            return "スキップ"
        if v >= 0.75:
            return "AIの可能性が高い"
        if v >= 0.40:
            return "判定が混在"
        return "人間が書いた可能性が高い"

    html = '<div style="font-family:sans-serif;max-width:800px">'

    for r in results:
        bg, bd = color(r["combined"])
        idx = r["index"] + 1
        html += (
            f'<div style="background:{bg};border-left:4px solid {bd}'
            f';padding:12px 16px;margin:8px 0;border-radius:8px">'
            f'<div style="display:flex;justify-content:space-between;margin-bottom:6px">'
            f'<span style="font-size:12px;color:#64748b"><b>段落 {idx}</b> — {label_text(r["combined"])}</span>'
            f'<span style="font-size:11px;color:#64748b">'
            f'統計:{fmt(r["stat"])} | Bino:{fmt(r["bino"])} | <b>総合:{fmt(r["combined"])}</b>'
            f'</span></div>'
            f'<p style="font-size:14px;color:#334155;line-height:1.7;margin:0;white-space:pre-wrap">'
            f'{r["text"]}</p></div>'
        )

    # 全体サマリー
    valid = [r for r in results if not r["skipped"]]
    if valid:
        avg_c = sum(r["combined"] for r in valid) / len(valid)
        avg_s = sum(r["stat"] for r in valid) / len(valid)
        avg_b = sum(r["bino"] for r in valid) / len(valid)
        ai_n = sum(1 for r in valid if r["label"] == "AI")
        mx_n = sum(1 for r in valid if r["label"] == "混在")
        hm_n = sum(1 for r in valid if r["label"] == "人間")
        abg, abd = color(avg_c)

        html += (
            f'<div style="background:{abg};border:2px solid #94a3b8;padding:16px'
            f';margin-top:20px;border-radius:12px">'
            f'<h3 style="margin:0 0 8px">全体の総合判定: {label_text(avg_c)} ({fmt(avg_c)})</h3>'
            f'<table style="font-size:14px">'
            f'<tr><td style="padding-right:20px">統計的手法（平均）</td><td><b>{fmt(avg_s)}</b></td></tr>'
            f'<tr><td>Binoculars（平均）</td><td><b>{fmt(avg_b)}</b></td></tr>'
            f'<tr><td>総合スコア（平均）</td><td><b>{fmt(avg_c)}</b></td></tr>'
            f'<tr><td>AI判定</td><td><b>{ai_n}</b> / {len(valid)}</td></tr>'
            f'<tr><td>混在判定</td><td><b>{mx_n}</b> / {len(valid)}</td></tr>'
            f'<tr><td>人間判定</td><td><b>{hm_n}</b> / {len(valid)}</td></tr>'
            f'</table>'
            f'<p style="font-size:11px;color:#64748b;margin-top:8px">'
            f'判定方式: Binoculars (70%) + 統計的手法 (30%)</p></div>'
        )

    html += "</div>"
    return html

# Gradio UI
demo = gr.Interface(
    fn=analyze_text,
    inputs=gr.Textbox(
        label="レポートのテキストを貼り付けてください",
        placeholder="ここにテキストを入力...\n\n段落は空行で区切ってください。",
        lines=12,
    ),
    outputs=gr.HTML(label="分析結果"),
    title="AI文章検出ツール",
    description="学生のレポートを段落ごとに分析し、AIが生成した可能性のある部分を検出します。\nBinoculars（LLMベース）と統計的手法のハイブリッド判定。",
    allow_flagging="never",
)

demo.launch(share=True, debug=False)